# Phase 5 — Applied Headcount Model and Scenarios

Companion notebook to `notes/phase5-headcount-model.md`. Builds a `HeadcountModel`, walks through the five scenarios, runs the sensitivity ranking, and computes the budget bridge in closed form.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.headcount import HeadcountModel, DTMCParams, BirthDeathParams

np.set_printoptions(precision=4, suppress=True)
sns.set_style('white')

## 1. Default model and the planner's questions

In [ ]:
m = HeadcountModel(n0=45)
print(f'E[n(12)]            = {m.expected_team_size(12):.2f}')
print(f'P(n(12) >= 50)      = {m.prob_reach_target(50, 12):.4f}')
print(f'P(n(12) >= 60)      = {m.prob_reach_target(60, 12):.4f}')
print(f'E[time -> n = 70]   = {m.expected_time_to_target(70):.2f} months')
print(f'E[time -> n = 80]   = {m.expected_time_to_target(80)} (asymptote)')
print('\nsteady-state composition:')
for k, v in m.steady_state_composition().items():
    print(f'  {k:<6} = {v:.4f}')

## 2. Scenario S1 — steady growth

In [ ]:
rows = []
for lam in [2.0, 2.2, 2.31, 2.5]:
    m_lam = HeadcountModel(
        n0=45,
        birth_death_params=BirthDeathParams(lambda_rate=lam, mu_rate=0.025),
    )
    rows.append(
        {
            'lambda': lam,
            'rho': lam / 0.025,
            'E[n(12)]': m_lam.expected_team_size(12),
            'P(n(12) >= 60)': m_lam.prob_reach_target(60, 12),
        }
    )
pd.DataFrame(rows).round(3)

## 3. Scenario S2 — hiring freeze

In [ ]:
freeze = HeadcountModel(
    n0=45, birth_death_params=BirthDeathParams(lambda_rate=0.0, mu_rate=0.025),
)
for thresh in [42, 40, 35, 30]:
    t = freeze.expected_time_to_target(thresh)
    print(f'E[time to drop below {thresh:>2}] = {t:.2f} months')

## 4. Scenario S3 — layoff recovery

In [ ]:
layoff = HeadcountModel(n0=35)
for target in [40, 45, 50, 60]:
    t = layoff.expected_time_to_target(target)
    print(f'E[time to reach {target}] = {t:.2f} months')

## 5. Scenario S4 — composition shift via Senior attrition

In [ ]:
rows = []
for p_se in [0.005, 0.0075, 0.01, 0.015, 0.02]:
    comp = HeadcountModel(
        dtmc_params=DTMCParams(p_se=p_se),
    ).steady_state_composition()
    rows.append({'p_se': p_se, **comp})
pd.DataFrame(rows).round(4)

## 6. Scenario S5 — seasonal hiring (variance bump)

Compose the CTMC quarter by quarter to compare a seasonal $\lambda(t)$ against the constant baseline.

In [ ]:
from src.ctmc import BirthDeathProcess

n_max, mu, n0, months = 200, 0.025, 45, 12
states = np.arange(n_max + 1)

def variance_at_horizon(quarter_lambdas):
    pi = np.zeros(n_max + 1); pi[n0] = 1.0
    for q in range(months):
        season = (q // 3) % 4
        lam = quarter_lambdas[season]
        bd_q = BirthDeathProcess.from_constant_hiring(lam, mu, n_max)
        pi = pi @ bd_q.transition_matrix(1.0)
    mean = (states * pi).sum()
    var = ((states - mean) ** 2 * pi).sum()
    return mean, var

mean_const, var_const = variance_at_horizon([2.0, 2.0, 2.0, 2.0])
mean_seas, var_seas = variance_at_horizon([3.0, 1.0, 3.0, 1.0])
print(f'constant lambda=2:  E[n(12)] = {mean_const:.2f}, Var = {var_const:.2f}')
print(f'seasonal:           E[n(12)] = {mean_seas:.2f}, Var = {var_seas:.2f}')
print(f'variance increase:  {(var_seas / var_const - 1) * 100:.2f}%')

## 7. Sensitivity tornado — by hand

In [ ]:
from dataclasses import replace

base_dtmc = DTMCParams()
base_senior = HeadcountModel().steady_state_composition()['Senior']

rows = []
for field, label in [
    ('p_jm', 'p_12 J->M'),
    ('p_je', 'p_14 J atr'),
    ('p_ms', 'p_23 M->S'),
    ('p_me', 'p_24 M atr'),
    ('p_se', 'p_34 S atr'),
]:
    base_value = float(getattr(base_dtmc, field))
    new_low = replace(base_dtmc, **{field: base_value * 0.5})
    new_high = replace(base_dtmc, **{field: base_value * 1.5})
    senior_low = HeadcountModel(dtmc_params=new_low).steady_state_composition()['Senior']
    senior_high = HeadcountModel(dtmc_params=new_high).steady_state_composition()['Senior']
    rows.append({'rate': label,
                 'pi_S @ -50%': senior_low,
                 'pi_S base': base_senior,
                 'pi_S @ +50%': senior_high,
                 '|delta|': max(abs(senior_low - base_senior), abs(senior_high - base_senior))})
df = pd.DataFrame(rows).sort_values('|delta|', ascending=False)
df.round(4)

## 8. Budget bridge — closed form

Using laws of total expectation/variance with $N$ = team size at month 12 and $S \sim \text{LogNormal}(9.2, 0.30)$ from Article 2.

In [ ]:
log_mean, log_sd = 9.2, 0.30
S_mean = np.exp(log_mean + log_sd**2 / 2)
S_var = (np.exp(log_sd**2) - 1) * np.exp(2 * log_mean + log_sd**2)
print(f'E[S]  = {S_mean:,.2f}')
print(f'sd(S) = {np.sqrt(S_var):,.2f}')

m_default = HeadcountModel(n0=45)
e_total, var_total = m_default.expected_total_salary(
    months=12, salary_mean=S_mean, salary_var=S_var
)
print(f'\nE[Total annual cost]   = {e_total / 1e6:,.3f} M')
print(f'sd(Total annual cost)  = {np.sqrt(var_total) / 1e6:,.3f} M')